# Task 5: Model Tuning

**Project:** Drug Review Classification — Condition Prediction  
**Objective:** Optimise the ensemble pipeline's hyperparameters to maximise classification accuracy, and demonstrate the improvement over the base model through rigorous evaluation.

---

## Overview

This notebook covers:
1. Tuning strategy and rationale
2. Hyperparameter search space — what each parameter controls
3. Randomised Search results and analysis
4. Focused Grid Search fine-tuning
5. Hyperparameter sensitivity analysis
6. Tuned model evaluation vs baseline
7. Learning curve analysis
8. Confusion matrix
9. Per-class performance discussion
10. Summary and conclusions

## 1. Setup

In [ ]:
import sys
import pickle
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, precision_recall_fscore_support
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

project_root = Path('..').resolve()
sys.path.insert(0, str(project_root))
from web.services.custom_transformers import (
    TextFeatureExtractor, SentimentFeatureExtractor, LearnedVocabularyExtractor
)

import __main__
__main__.TextFeatureExtractor     = TextFeatureExtractor
__main__.SentimentFeatureExtractor = SentimentFeatureExtractor
__main__.LearnedVocabularyExtractor = LearnedVocabularyExtractor

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
PALETTE = ['#14b8a6', '#eaa0a2', '#67cbdc', '#3a4664', '#cbcbcb']

print('Setup complete.')

In [ ]:
# Load data
df_train = pd.read_csv('../data/processed/cleaned_train_data.csv')
df_test  = pd.read_csv('../data/processed/cleaned_test_data.csv')

X_train = df_train['review'].fillna('').values
y_train = df_train['condition'].values
X_test  = df_test['review'].fillna('').values
y_test  = df_test['condition'].values

label_encoder = joblib.load('../models/label_encoder.pkl')
y_train_enc = label_encoder.transform(y_train)
y_test_enc  = label_encoder.transform(y_test)

# Load tuned model
with open('../models/tuned_pipeline.pkl', 'rb') as f:
    tuned_model = pickle.load(f)

# Load tuning results
with open('../models/tuning_results.json') as f:
    tuning_results = json.load(f)

print(f'Training samples : {len(X_train):,}')
print(f'Test samples     : {len(X_test):,}')
print(f'Classes          : {label_encoder.classes_.tolist()}')
print(f'Tuning date      : {tuning_results["tuning_date"][:10]}')

## 2. Tuning Strategy

### Why hyperparameter tuning?

The base ensemble in Task 4 used default or manually set hyperparameters. While it achieved ~94.6% accuracy, the model's performance depends heavily on choices such as:
- How many TF-IDF features to keep
- Whether to include bigrams
- How deep the Random Forest trees should grow
- The learning rate and depth of XGBoost trees
- The relative weight given to each model in the ensemble vote

Searching over these systematically finds combinations that work better together than any manual choice.

### Two-stage search strategy

A two-stage approach was used to balance thoroughness with computational cost:

| Stage | Method | Purpose | Iterations |
|-------|--------|---------|------------|
| Stage 1 | **Randomised Search** | Broad exploration of the parameter space | 20 random combinations × 3 folds = 60 fits |
| Stage 2 | **Grid Search** | Fine-tuning around the best values found | ~6 combinations × 3 folds = 18 fits |

**Why Randomised Search first?** A full grid search over all parameters would require thousands of fits (prohibitively slow). Randomised Search samples randomly from the full space, finding promising regions efficiently. Grid Search then refines within that region.

**Why 3-fold CV?** Stratified 3-fold cross-validation was used to ensure each fold contains proportional class representation, addressing the class imbalance identified in Task 1.

## 3. Hyperparameter Search Space — What Each Parameter Controls

In [ ]:
param_descriptions = pd.DataFrame([
    # TF-IDF
    {'Parameter': 'tfidf__max_features', 'Values tested': '[1500, 2000, 2500]',
     'What it controls': 'How many of the most frequent terms to keep in the vocabulary',
     'Impact': 'More features = richer vocabulary but slower training and risk of noise'},
    {'Parameter': 'tfidf__ngram_range', 'Values tested': '[(1,1), (1,2)]',
     'What it controls': 'Whether single words only, or word pairs (bigrams) are included',
     'Impact': 'Bigrams capture phrases like "blood pressure" and "feeling hopeless"'},
    {'Parameter': 'tfidf__min_df', 'Values tested': '[2, 3]',
     'What it controls': 'Minimum document frequency — rare terms below this are dropped',
     'Impact': 'Higher values reduce noise from typos and very rare words'},
    {'Parameter': 'tfidf__max_df', 'Values tested': '[0.80, 0.85]',
     'What it controls': 'Maximum document frequency — very common terms are dropped',
     'Impact': 'Removes stop-like words that appear in almost all reviews'},
    # Logistic Regression
    {'Parameter': 'lr__C', 'Values tested': '[0.1, 1.0, 10.0]',
     'What it controls': 'Inverse regularisation strength — lower C = stronger regularisation',
     'Impact': 'Controls overfitting; high C allows the model to fit the training data more closely'},
    # Random Forest
    {'Parameter': 'rf__n_estimators', 'Values tested': '[100, 150, 200]',
     'What it controls': 'Number of decision trees in the forest',
     'Impact': 'More trees = more stable predictions but slower training'},
    {'Parameter': 'rf__max_depth', 'Values tested': '[15, 20, None]',
     'What it controls': 'Maximum depth each tree can grow',
     'Impact': 'Deeper trees capture more complex patterns but risk overfitting'},
    # XGBoost
    {'Parameter': 'xgb__n_estimators', 'Values tested': '[100, 150, 200]',
     'What it controls': 'Number of boosting rounds',
     'Impact': 'More rounds = better correction of errors but slower and risk of overfitting'},
    {'Parameter': 'xgb__max_depth', 'Values tested': '[3, 5, 7]',
     'What it controls': 'Maximum depth of each boosted tree',
     'Impact': 'Deeper trees capture more interactions; shallower trees generalise better'},
    {'Parameter': 'xgb__learning_rate', 'Values tested': '[0.05, 0.1, 0.2]',
     'What it controls': 'Step size for each boosting round',
     'Impact': 'Lower rate = slower learning but better generalisation; higher rate = faster but may overshoot'},
    # Ensemble
    {'Parameter': 'classifier__weights', 'Values tested': '[[1,1,1],[1,2,2],[1,2,3],[1,3,3],[1,3,4]]',
     'What it controls': 'Relative influence of LR, RF, XGB in the final vote',
     'Impact': 'Upweighting stronger models improves overall predictions'},
])

param_descriptions

## 4. Tuning Results Summary

In [ ]:
print('=' * 60)
print('TUNING RESULTS SUMMARY')
print('=' * 60)
print(f"""
  Randomised Search best CV  : {tuning_results['random_search_best_cv']:.4f}
  Grid Search best CV        : {tuning_results['grid_search_best_cv']:.4f}

  Baseline accuracy          : {tuning_results['baseline_accuracy']:.4f}  ({tuning_results['baseline_accuracy']*100:.2f}%)
  Tuned model accuracy       : {tuning_results['test_accuracy']:.4f}  ({tuning_results['test_accuracy']*100:.2f}%)
  Improvement                : +{tuning_results['improvement']:.4f}  (+{tuning_results['improvement_pct']:.2f}%)

  Precision                  : {tuning_results['test_precision']:.4f}
  Recall                     : {tuning_results['test_recall']:.4f}
  F1 Score                   : {tuning_results['test_f1']:.4f}

  5-Fold CV mean             : {tuning_results['cv_5fold_mean']:.4f} ± {tuning_results['cv_5fold_std']:.4f}

  Total tuning time          : {tuning_results['total_time_minutes']:.1f} minutes
""")

print('Best hyperparameters found:')
for k, v in tuning_results['best_params'].items():
    print(f'  {k:<45} = {v}')

## 5. Before vs After Tuning — Visual Comparison

In [ ]:
# Metrics comparison
base_metrics = {
    'Accuracy':  0.9463,
    'Precision': 0.9476,
    'Recall':    0.9463,
    'F1 Score':  0.9454,
}
tuned_metrics = {
    'Accuracy':  tuning_results['test_accuracy'],
    'Precision': tuning_results['test_precision'],
    'Recall':    tuning_results['test_recall'],
    'F1 Score':  tuning_results['test_f1'],
}
baseline_acc = tuning_results['baseline_accuracy']

metrics = list(base_metrics.keys())
x = np.arange(len(metrics))
width = 0.25

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grouped bar chart
bars1 = axes[0].bar(x - width, [baseline_acc] * 4, width,
                    label='Baseline (TF-IDF + LR)', color='#cbd5e1',
                    edgecolor='white')
bars2 = axes[0].bar(x, list(base_metrics.values()), width,
                    label='Base Ensemble', color='#67cbdc',
                    edgecolor='white')
bars3 = axes[0].bar(x + width, list(tuned_metrics.values()), width,
                    label='Tuned Ensemble ★', color='#14b8a6',
                    edgecolor='white')

axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics, fontsize=11)
axes[0].set_ylim(0.88, 1.01)
axes[0].set_title('Performance: Baseline vs Base vs Tuned', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Score', fontsize=11)
axes[0].legend(fontsize=10)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.001,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=8, rotation=45)

# Improvement waterfall
stages = ['Baseline\n(TF-IDF+LR)', 'Base\nEnsemble', 'Tuned\nEnsemble']
values = [baseline_acc, base_metrics['Accuracy'], tuned_metrics['Accuracy']]
colors_wf = ['#cbd5e1', '#67cbdc', '#14b8a6']
bars_wf = axes[1].bar(stages, values, color=colors_wf, edgecolor='white', width=0.5)
axes[1].set_ylim(0.88, 1.01)
axes[1].set_title('Accuracy Progression', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Test Accuracy', fontsize=11)
for bar, val in zip(bars_wf, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.001,
                 f'{val:.4f}\n({val*100:.2f}%)',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')

# Annotate improvement arrows
for i in range(len(values)-1):
    diff = values[i+1] - values[i]
    sign = '+' if diff >= 0 else ''
    mid_x = i + 0.75
    axes[1].annotate(f'{sign}{diff:.4f}',
                     xy=(mid_x, (values[i]+values[i+1])/2),
                     fontsize=10, color='#134e4a', fontweight='bold',
                     ha='center')

plt.tight_layout()
plt.savefig('../outputs/figures/tuning_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: outputs/figures/tuning_comparison.png')

## 6. Hyperparameter Sensitivity Analysis

The sensitivity analysis shows how mean CV accuracy varied for each hyperparameter value across the 20 Randomised Search iterations. This tells us which parameters had the most impact on performance.

In [ ]:
# Load the pre-generated sensitivity plot
from IPython.display import Image, display

sensitivity_path = Path('../outputs/figures/hyperparameter_sensitivity.png')
if sensitivity_path.exists():
    display(Image(str(sensitivity_path)))
    print('Hyperparameter sensitivity plot loaded from outputs/figures/')
else:
    print('Run scripts/tune_model.py first to generate this plot.')

### Sensitivity Analysis — Findings

From the sensitivity plots:

| Parameter | Finding |
|-----------|--------|
| **TF-IDF max features** | 2500 features outperforms 1500 — more vocabulary gives the model richer signal |
| **XGB max_depth** | Depth 7 outperforms depth 3 — deeper trees capture more complex symptom patterns |
| **XGB learning_rate** | 0.2 marginally better than 0.05 — faster learning works well when n_estimators is also high |
| **RF n_estimators** | Diminishing returns beyond 150 — the forest stabilises quickly on this dataset |
| **LR C** | C=10 (less regularisation) works best — the TF-IDF features are informative, not noisy |
| **Ensemble weights** | Upweighting XGB ([1,2,3]) consistently outperforms equal weights — XGB's higher accuracy deserves more influence |

## 7. Tuned Model Evaluation

In [ ]:
y_pred  = tuned_model.predict(X_test)
y_proba = tuned_model.predict_proba(X_test)

accuracy  = accuracy_score(y_test_enc, y_pred)
p, r, f1, _ = precision_recall_fscore_support(
    y_test_enc, y_pred, average='weighted'
)

print('=' * 55)
print('TUNED MODEL — TEST SET PERFORMANCE')
print('=' * 55)
print(f'  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)')
print(f'  Precision : {p:.4f}')
print(f'  Recall    : {r:.4f}')
print(f'  F1 Score  : {f1:.4f}')

print(f'\n  Improvement over baseline: +{accuracy - baseline_acc:.4f} (+{(accuracy-baseline_acc)/baseline_acc*100:.2f}%)')

print('\nPer-class report:')
print(classification_report(
    y_test_enc, y_pred,
    target_names=label_encoder.classes_,
    digits=4
))

## 8. Confusion Matrix — Tuned Model

In [ ]:
cm      = confusion_matrix(y_test_enc, y_pred)
cm_norm = confusion_matrix(y_test_enc, y_pred, normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            ax=axes[0], linewidths=0.5)
axes[0].set_title(f'Confusion Matrix — Tuned Model (Accuracy: {accuracy:.2%})',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted', fontsize=11)
axes[0].set_ylabel('Actual', fontsize=11)
axes[0].tick_params(axis='x', rotation=20)

sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix — Normalised (Recall per class)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted', fontsize=11)
axes[1].set_ylabel('Actual', fontsize=11)
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

### Per-Class Performance Discussion

| Condition | Precision | Recall | F1 | Analysis |
|-----------|-----------|--------|-----|----------|
| **Depression** | 97.60% | 99.07% | 98.33% | Highest recall — the model rarely misses a Depression case. The large volume of training data (9,154 samples) and highly distinctive vocabulary (medication names like Sertraline, Escitalopram, emotional language) make this the easiest class to identify. |
| **Diabetes, Type 2** | 97.95% | 94.80% | 96.35% | High precision — when the model predicts Diabetes, it is nearly always correct. Slightly lower recall means some Diabetes reviews are occasionally predicted as High Blood Pressure, likely because both conditions involve medication management language. |
| **High Blood Pressure** | 95.98% | 93.52% | 94.74% | The most challenging class — lowest recall of the three. Both High BP and Diabetes involve terms like "blood", "doctor", "medication", "side effects" without the distinctive emotional vocabulary of Depression. The minority class status (2,403 samples) also contributes. |

The **macro F1 of 96.47%** confirms strong performance across all three classes, not just the majority class.

## 9. Learning Curve Analysis

In [ ]:
from IPython.display import Image, display

lc_path = Path('../outputs/figures/learning_curve_tuned.png')
if lc_path.exists():
    display(Image(str(lc_path)))
else:
    print('Run scripts/tune_model.py first to generate this plot.')

### Learning Curve — Findings

The learning curve plots training accuracy and validation accuracy as the number of training samples increases from 10% to 100% of the training set.

**What to look for:**
- If training accuracy >> validation accuracy → **overfitting** (model memorises training data)
- If both are low → **underfitting** (model too simple)
- If both converge at a high value → **good fit** (model generalises well)

**Observations from the curve:**
1. Training and validation accuracy converge as training size increases — the model generalises well
2. The gap between training and validation is small (< 2%) at full dataset size — no significant overfitting
3. Validation accuracy continues to improve with more data — suggesting the model would benefit from even more training samples if available
4. The narrow confidence band (shaded area) indicates low variance across CV folds — stable, reproducible performance

## 10. Cross-Validation Stability

In [ ]:
cv_scores = tuning_results['cv_5fold_scores']
cv_mean   = tuning_results['cv_5fold_mean']
cv_std    = tuning_results['cv_5fold_std']

fig, ax = plt.subplots(figsize=(9, 5))

folds = [f'Fold {i+1}' for i in range(len(cv_scores))]
bars = ax.bar(folds, cv_scores, color=PALETTE[0],
              edgecolor='white', width=0.5)
ax.axhline(cv_mean, color='#ef4444', linestyle='--', linewidth=2,
           label=f'Mean: {cv_mean:.4f}')
ax.axhline(accuracy, color='#3b82f6', linestyle='--', linewidth=2,
           label=f'Test accuracy: {accuracy:.4f}')
ax.fill_between(range(-1, len(folds)+1),
                cv_mean - cv_std, cv_mean + cv_std,
                alpha=0.1, color='#ef4444',
                label=f'± 1 std ({cv_std:.4f})')
ax.set_xlim(-0.5, len(folds) - 0.5)
ax.set_ylim(0.93, 1.0)
ax.set_title('5-Fold Cross-Validation — Tuned Model', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=11)
ax.legend(fontsize=10)
for bar, val in zip(bars, cv_scores):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/figures/cv_tuned.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'CV scores   : {[f"{s:.4f}" for s in cv_scores]}')
print(f'Mean ± Std  : {cv_mean:.4f} ± {cv_std:.4f}')
print(f'Test acc    : {accuracy:.4f}')
print(f'Gap(test-CV): {accuracy - cv_mean:+.4f}')

## 11. OOD Confidence Thresholds

In [ ]:
with open('../models/ood_statistics.json') as f:
    ood_stats = json.load(f)

thresholds = ood_stats['confidence_thresholds']

print('OOD Confidence Thresholds (learned from tuned model)')
print('-' * 50)
for level, value in thresholds.items():
    print(f'  {level:<10} : {value:.3f} ({value*100:.1f}%)')

max_proba = np.array(y_proba).max(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(max_proba, bins=50, color='#14b8a6', edgecolor='white', alpha=0.8,
        label='Prediction confidence')

threshold_colors = {'very_low': '#ef4444', 'low': '#f97316',
                    'medium': '#eab308', 'high': '#22c55e'}
for level, color in threshold_colors.items():
    val = thresholds[level]
    ax.axvline(val, color=color, linestyle='--', linewidth=1.8,
               label=f'{level.replace("_"," ").title()}: {val:.3f}')

ax.set_title('Prediction Confidence vs OOD Thresholds', fontsize=13, fontweight='bold')
ax.set_xlabel('Max Class Probability', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.legend(fontsize=9, loc='upper left')

plt.tight_layout()
plt.savefig('../outputs/figures/ood_thresholds.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nMost predictions ({(max_proba >= thresholds["high"])*100:.0f}% on test set) '
      f'exceed the high confidence threshold.')

### OOD Threshold Interpretation

The confidence thresholds are computed from the **tuned model's predictions on the training set** (percentiles of the max class probability distribution). This ensures the thresholds reflect the model's actual behaviour rather than being arbitrarily set.

| Level | Threshold | Meaning |
|-------|-----------|--------|
| Very Low | 68.2% | Below this — likely out-of-distribution input |
| Low | 78.2% | Low confidence — model uncertain |
| Medium | 85.4% | Moderate confidence |
| High | 91.3% | High confidence — reliable prediction |

## 12. Final Summary — Task 5 Conclusions

### What tuning achieved

In [ ]:
summary = pd.DataFrame([
    {'Stage': 'Baseline (TF-IDF + Logistic Regression)',
     'Accuracy': f"{tuning_results['baseline_accuracy']:.4f}",
     'F1 (weighted)': 'N/A',
     'CV Mean': 'N/A',
     'Notes': 'Simple single-model baseline'},
    {'Stage': 'Base Ensemble (Task 4)',
     'Accuracy': '0.9463',
     'F1 (weighted)': '0.9454',
     'CV Mean': '~0.945',
     'Notes': 'LR + RF + XGB with default params'},
    {'Stage': 'Tuned Ensemble (Task 5) ★',
     'Accuracy': f"{tuning_results['test_accuracy']:.4f}",
     'F1 (weighted)': f"{tuning_results['test_f1']:.4f}",
     'CV Mean': f"{tuning_results['cv_5fold_mean']:.4f} ± {tuning_results['cv_5fold_std']:.4f}",
     'Notes': 'RandomisedSearch + GridSearch optimised'},
])

print('=== PROGRESSION SUMMARY ===')
summary

### Key conclusions

1. **Tuning produced a significant improvement** — from 94.63% (base ensemble) to **97.39%** (+2.76 percentage points), representing a 51% reduction in the error rate

2. **The most impactful parameters were:**
   - `tfidf__ngram_range = (1,2)` — including bigrams captures medical phrases that unigrams miss
   - `xgb__max_depth = 7` — deeper trees capture the complex co-occurrence patterns in medical reviews
   - `xgb__learning_rate = 0.2` — faster convergence when combined with 200 estimators
   - `classifier__weights = [1, 2, 3]` — correctly upweighting XGBoost's superior individual performance

3. **The learning curve confirms no overfitting** — training and validation curves converge, and the test accuracy (97.39%) is consistent with the 5-fold CV mean (96.61%)

4. **Depression classification is excellent** (F1 = 98.33%) due to its distinctive vocabulary and volume of training data. High Blood Pressure is the hardest class (F1 = 94.74%) due to vocabulary overlap with Diabetes. Both minority classes (Diabetes, High BP) achieved over 94% F1, confirming the `class_weight='balanced'` strategy was effective.

5. **The model is deployment-ready** — the tuned pipeline is saved as `models/tuned_pipeline.pkl` and integrated into the Flask web application with OOD detection to safely reject out-of-scope inputs.